#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

#### Loading the Processed Training and Testing Datasets

In [2]:
processed_dir = Path("../data/processed")

train_df = pd.read_csv(processed_dir / "train.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

X_train = train_df["utterance"]
X_test = test_df["utterance"]

y_train = train_df["intent"]
y_test = test_df["intent"]

print(f"Training Dataset Shape: {train_df.shape}")
print(f"Testing Dataset Shape: {test_df.shape}")

print(f"\nTraining Samples: {len(X_train):,}")
print(f"Testing Samples: {len(X_test):,}")

print(f"\nTraining Classes: {y_train.nunique()}")
print(f"Testing Classes: {y_test.nunique()}")

display(train_df.head())

Training Dataset Shape: (17210, 2)
Testing Dataset Shape: (4303, 2)

Training Samples: 17,210
Testing Samples: 4,303

Training Classes: 27
Testing Classes: 27


,utterance,intent
0,can you track my delivery?,track_order
1,"im discontent with the service, i need help su...",complaint
2,how to change my profile,edit_account
3,how to check the cancellation fee,check_cancellation_fee
4,where do I notify an issue paying?,payment_issue


#### 2. Word-Level Feature Engineering

#### Creating Word-Level TF-IDF Features

In [3]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=True,
    strip_accents="unicode",
    dtype=np.float32
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

print(f"Training Feature Matrix: {X_train_word.shape}")
print(f"Testing Feature Matrix: {X_test_word.shape}")
print(f"Vocabulary Size: {len(word_vectorizer.vocabulary_):,}")

print(f"\nTraining Matrix Sparsity: {1 - X_train_word.nnz / (X_train_word.shape[0] * X_train_word.shape[1]):.4%}")
print(f"Testing Matrix Sparsity: {1 - X_test_word.nnz / (X_test_word.shape[0] * X_test_word.shape[1]):.4%}")

Training Feature Matrix: (17210, 2749)
Testing Feature Matrix: (4303, 2749)
Vocabulary Size: 2,749

Training Matrix Sparsity: 99.3865%
Testing Matrix Sparsity: 99.3905%


#### Observations
- **2,749 word-level TF-IDF features** were generated using unigram and bigram representations.
- **Training and testing matrices contain 17,210 and 4,303 samples**, respectively, with consistent feature dimensions.
- **Key Finding:** Both matrices exhibit approximately 99.39% sparsity, supporting the use of sparse matrix storage and computationally efficient classifiers.

#### 3. Character-Level Feature Engineering

#### Creating Character-Level TF-IDF Features

In [4]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=True,
    strip_accents="unicode",
    dtype=np.float32
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

print(f"Training Feature Matrix: {X_train_char.shape}")
print(f"Testing Feature Matrix: {X_test_char.shape}")
print(f"Vocabulary Size: {len(char_vectorizer.vocabulary_):,}")

print(f"\nTraining Matrix Sparsity: {1 - X_train_char.nnz / (X_train_char.shape[0] * X_train_char.shape[1]):.4%}")
print(f"Testing Matrix Sparsity: {1 - X_test_char.nnz / (X_test_char.shape[0] * X_test_char.shape[1]):.4%}")

Training Feature Matrix: (17210, 6190)
Testing Feature Matrix: (4303, 6190)
Vocabulary Size: 6,190

Training Matrix Sparsity: 98.4799%
Testing Matrix Sparsity: 98.4852%


#### Observations
- **6,190 character-level TF-IDF features** were generated using character n-grams of lengths 3–5.
- **Training and testing matrices retain consistent dimensions**, with sparsity levels of 98.48% and 98.49%, respectively.
- **Key Finding:** Character-level features provide a complementary representation of customer utterances, potentially capturing spelling variations and word fragments missed by word-level TF-IDF.

#### 4. Combined Feature Engineering

#### Combining Word-Level and Character-Level TF-IDF Features

In [5]:
X_train_combined = sparse.hstack(
    [X_train_word, X_train_char],
    format="csr",
    dtype=np.float32
)

X_test_combined = sparse.hstack(
    [X_test_word, X_test_char],
    format="csr",
    dtype=np.float32
)

print(f"Combined Training Matrix: {X_train_combined.shape}")
print(f"Combined Testing Matrix: {X_test_combined.shape}")

print(f"\nWord-Level Features: {X_train_word.shape[1]:,}")
print(f"Character-Level Features: {X_train_char.shape[1]:,}")
print(f"Total Combined Features: {X_train_combined.shape[1]:,}")

print(f"\nCombined Training Sparsity: {1 - X_train_combined.nnz / (X_train_combined.shape[0] * X_train_combined.shape[1]):.4%}")
print(f"Combined Testing Sparsity: {1 - X_test_combined.nnz / (X_test_combined.shape[0] * X_test_combined.shape[1]):.4%}")

Combined Training Matrix: (17210, 8939)
Combined Testing Matrix: (4303, 8939)

Word-Level Features: 2,749
Character-Level Features: 6,190
Total Combined Features: 8,939

Combined Training Sparsity: 98.7587%
Combined Testing Sparsity: 98.7636%


#### Observations
- **8,939 combined TF-IDF features** were generated by integrating 2,749 word-level and 6,190 character-level features.
- **Training and testing matrices retain 17,210 and 4,303 samples**, respectively, with consistent feature dimensions.
- **Key Finding:** The combined representation captures complementary word and character patterns while maintaining approximately 98.76% sparsity, making it suitable for efficient sparse-text classification.

#### 5. Saving Feature Engineering Artifacts

#### Saving TF-IDF Feature Matrices and Fitted Vectorizers

In [6]:
features_dir = Path("../data/features")
models_dir = Path("../models")

features_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

feature_matrices = {
    "X_train_word.npz": X_train_word,
    "X_test_word.npz": X_test_word,
    "X_train_char.npz": X_train_char,
    "X_test_char.npz": X_test_char,
    "X_train_combined.npz": X_train_combined,
    "X_test_combined.npz": X_test_combined
}

for filename, matrix in feature_matrices.items():
    sparse.save_npz(features_dir / filename, matrix)

joblib.dump(word_vectorizer, models_dir / "word_tfidf_vectorizer.joblib")
joblib.dump(char_vectorizer, models_dir / "char_tfidf_vectorizer.joblib")

print("Feature matrices and fitted vectorizers saved successfully.\n")

for filename in feature_matrices:
    filepath = features_dir / filename
    print(f"{filename}: {filepath.stat().st_size / (1024 ** 2):.2f} MB")

print("\nSaved Vectorizers:")
print("word_tfidf_vectorizer.joblib")
print("char_tfidf_vectorizer.joblib")

Feature matrices and fitted vectorizers saved successfully.

X_train_word.npz: 1.24 MB
X_test_word.npz: 0.30 MB
X_train_char.npz: 5.36 MB
X_test_char.npz: 1.46 MB
X_train_combined.npz: 6.86 MB
X_test_combined.npz: 1.82 MB

Saved Vectorizers:
word_tfidf_vectorizer.joblib
char_tfidf_vectorizer.joblib


#### Observations
- **Six TF-IDF feature matrices** were successfully saved, covering word-level, character-level, and combined representations for both training and testing.
- **Both fitted TF-IDF vectorizers** were saved separately, allowing consistent transformation of new customer utterances without refitting.
- **Decision:** Saved feature matrices and vectorizers will be reused during model development, evaluation, and explainability to ensure reproducibility.

#### Verifying Saved Feature Matrices and Vectorizers

In [7]:
loaded_matrices = {
    filename: sparse.load_npz(features_dir / filename)
    for filename in feature_matrices
}

loaded_word_vectorizer = joblib.load(
    models_dir / "word_tfidf_vectorizer.joblib"
)

loaded_char_vectorizer = joblib.load(
    models_dir / "char_tfidf_vectorizer.joblib"
)

for filename, matrix in loaded_matrices.items():
    print(f"{filename}: {matrix.shape}")

print(f"\nLoaded Word Vocabulary: {len(loaded_word_vectorizer.vocabulary_):,}")
print(f"Loaded Character Vocabulary: {len(loaded_char_vectorizer.vocabulary_):,}")

print("\nArtifact Validation:")
print(f"Word Features Match: {loaded_matrices['X_train_word.npz'].shape[1] == len(loaded_word_vectorizer.vocabulary_)}")
print(f"Character Features Match: {loaded_matrices['X_train_char.npz'].shape[1] == len(loaded_char_vectorizer.vocabulary_)}")
print(f"Combined Features Match: {loaded_matrices['X_train_combined.npz'].shape[1] == len(loaded_word_vectorizer.vocabulary_) + len(loaded_char_vectorizer.vocabulary_)}")

X_train_word.npz: (17210, 2749)
X_test_word.npz: (4303, 2749)
X_train_char.npz: (17210, 6190)
X_test_char.npz: (4303, 6190)
X_train_combined.npz: (17210, 8939)
X_test_combined.npz: (4303, 8939)

Loaded Word Vocabulary: 2,749
Loaded Character Vocabulary: 6,190

Artifact Validation:
Word Features Match: True
Character Features Match: True
Combined Features Match: True


#### Observations
- **All six saved feature matrices were reloaded successfully**, preserving their original training and testing dimensions.
- **Both fitted vectorizers retained their vocabularies**, containing 2,749 word-level and 6,190 character-level features.
- **Key Finding:** All artifact consistency checks passed, confirming that the saved feature representations are ready for reproducible model development and evaluation.